In [ ]:
!pip uninstall -y sklearn-compat ibis-framework imbalanced-learn google-genai
!pip install polars==1.30.0
# === GOOGLE GEMINI ===
#!pip install fenic[google]
# === ANTHROPIC CLAUDE ===
#!pip install fenic[anthropic]
# === OPENAI (Default) ===
!pip install fenic

In [ ]:
import os 
import getpass

# 🔌 MULTI-PROVIDER SETUP - Choose your preferred LLM provider
# Uncomment ONE of the provider sections below:

# === OPENAI (Default) ===
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

# === GOOGLE GEMINI ===
# os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API Key:")

# === ANTHROPIC CLAUDE ===
# os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key:")

# 🛠️ Custom UDFs with Semantic Context

**Hook:** *"Extend Fenic with your domain expertise - AI meets business logic"*

Every business has unique logic - credit scoring models, compliance rules, domain-specific calculations. Fenic's semantic UDFs let you combine traditional programming with AI understanding. Write functions that leverage both structured data and semantic context for powerful business workflows.

**What you'll see in this 2-minute demo:**
- 🧠 **Semantic-aware functions** - UDFs that understand context and meaning
- 🏦 **Business rule integration** - Combine AI with traditional logic
- 📊 **Complex decision making** - Multi-factor analysis with semantic input
- 🔧 **Extensible framework** - Build domain-specific AI functions

Perfect for financial services, healthcare, legal tech, and specialized business domains.

In [ ]:
import fenic as fc
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
import json

# 🛠️ Configure session for custom UDF development
session = fc.Session.get_or_create(fc.SessionConfig(
    app_name="custom_udfs_demo",
    semantic=fc.SemanticConfig(
        language_models={
            "domain_expert": fc.OpenAILanguageModel(model_name="gpt-4o-mini", rpm=500, tpm=200_000),
            # "domain_expert": fc.GoogleDeveloperLanguageModel(model_name="gemini-2.5-flash-lite", rpm=1000, tpm=1_000_000),
            # "domain_expert": fc.AnthropicLanguageModel(model_name="claude-3-5-sonnet-20241022", rpm=500, tpm=200_000),
            "business_analyst": fc.OpenAILanguageModel(model_name="gpt-4o", rpm=100, tpm=50_000)
        },
        default_language_model="business_analyst"
    )
))

print("✅ Custom UDF development session configured")
print("   • Domain Expert: GPT-4o-mini for fast semantic analysis")
print("   • Business Analyst: GPT-4o for complex reasoning")

## 🏦 Step 1: Business Domain Example - Loan Risk Assessment

Create a realistic financial dataset requiring complex business logic:

In [ ]:
# 🏦 Loan application dataset with structured and unstructured data
loan_applications = session.create_dataframe([
    {
        "application_id": "APP001",
        "applicant_name": "Sarah Johnson",
        "credit_score": 720,
        "annual_income": 85000,
        "loan_amount": 250000,
        "employment_years": 3.5,
        "debt_to_income": 0.28,
        "application_notes": "Stable employment as software engineer at tech startup. Recently promoted to senior role. Planning to purchase first home with spouse. Has savings for 20% down payment. No previous bankruptcies or major financial issues."
    },
    {
        "application_id": "APP002",
        "applicant_name": "Mike Rodriguez", 
        "credit_score": 680,
        "annual_income": 120000,
        "loan_amount": 400000,
        "employment_years": 8.2,
        "debt_to_income": 0.35,
        "application_notes": "Experienced marketing director with strong track record. Recently divorced, paying alimony which affects DTI ratio. Has significant stock options that will vest next year. Previous foreclosure 7 years ago due to job loss during economic downturn."
    },
    {
        "application_id": "APP003",
        "applicant_name": "Jennifer Chen",
        "credit_score": 780,
        "annual_income": 95000,
        "loan_amount": 180000,
        "employment_years": 12.0,
        "debt_to_income": 0.22,
        "application_notes": "Government employee with excellent job security. Military veteran with VA benefits. Refinancing existing mortgage to take advantage of lower rates. Excellent payment history and no negative marks on credit report."
    },
    {
        "application_id": "APP004",
        "applicant_name": "David Park",
        "credit_score": 650,
        "annual_income": 75000,
        "loan_amount": 300000,
        "employment_years": 2.1,
        "debt_to_income": 0.42,
        "application_notes": "Self-employed contractor with variable income. Business has grown significantly in past 2 years. High DTI due to business expansion loans. Filed Chapter 13 bankruptcy 5 years ago but has been current on all payments since."
    },
    {
        "application_id": "APP005",
        "applicant_name": "Lisa Williams",
        "credit_score": 740,
        "annual_income": 110000,
        "loan_amount": 275000,
        "employment_years": 6.8,
        "debt_to_income": 0.31,
        "application_notes": "Healthcare professional with stable income. Recently inherited assets from family estate. Planning investment property purchase. Has experience as landlord with existing rental property generating positive cash flow."
    }
])

print("🏦 Loan Application Dataset:")
print("   • Mix of structured (credit score, income) and unstructured (notes) data")
print("   • Complex financial situations requiring nuanced assessment")
print("   • Perfect for demonstrating semantic UDFs")
loan_applications.select("application_id", "applicant_name", "credit_score", "loan_amount").show()

## 🧠 Step 2: Semantic Risk Analysis Schema

Define the AI-powered risk assessment output:

In [ ]:
# 🧠 Risk assessment output schema
class RiskAssessment(BaseModel):
    stability_factors: List[str] = Field(description="Positive stability indicators from notes")
    risk_factors: List[str] = Field(description="Potential risk indicators from notes")
    employment_stability: str = Field(description="Assessment: excellent, good, fair, poor")
    financial_complexity: str = Field(description="Situation complexity: simple, moderate, complex, very_complex")
    narrative_risk_score: float = Field(description="Risk score from narrative 0.0-10.0 (lower = better)")
    special_considerations: List[str] = Field(description="Unique factors requiring attention")
    confidence_level: float = Field(description="Assessment confidence 0.0-1.0")

print("🧠 Risk Assessment Schema:")
print("   • stability_factors: Positive indicators from application notes")
print("   • risk_factors: Concerning elements identified")
print("   • employment_stability: Job security assessment")
print("   • financial_complexity: Situation difficulty level")
print("   • narrative_risk_score: AI-derived risk (0-10 scale)")
print("   • special_considerations: Unique factors for manual review")
print("   • confidence_level: AI certainty in assessment")

## 🔧 Step 3: Custom UDF - Advanced Risk Scoring

Create domain-specific functions that combine structured data with semantic analysis:

In [ ]:
# 🔧 Custom UDF: Advanced Risk Calculator
@fc.udf(return_type=fc.StructType([
    fc.StructField("composite_score", fc.DoubleType),
    fc.StructField("risk_category", fc.StringType),
    fc.StructField("recommendation", fc.StringType),
    fc.StructField("credit_component", fc.DoubleType),
    fc.StructField("dti_component", fc.DoubleType),
    fc.StructField("employment_component", fc.DoubleType),
    fc.StructField("lti_component", fc.DoubleType),
    fc.StructField("narrative_component", fc.DoubleType)
]))
def calculate_comprehensive_risk(
    credit_score: int,
    debt_to_income: float, 
    employment_years: float,
    loan_to_income_ratio: float,
    narrative_risk: float
) -> Dict[str, Any]:
    """
    Custom UDF that combines traditional underwriting metrics with AI narrative analysis
    """
    # Traditional risk scoring weights
    credit_weight = 0.35
    dti_weight = 0.25 
    employment_weight = 0.15
    lti_weight = 0.15
    narrative_weight = 0.10
    
    # Normalize credit score (300-850 range) - ensure float
    credit_score_normalized = float(max(0, min(10, (credit_score - 300) / 55)))
    
    # DTI risk (lower is better, invert scale) - ensure float
    dti_risk = float(min(10, debt_to_income * 20))  # 0.5 DTI = 10 risk
    dti_score = float(10 - dti_risk)
    
    # Employment stability (longer is better) - ensure float
    employment_score = float(min(10, employment_years * 2))  # 5+ years = 10
    
    # Loan to income risk - ensure float
    lti_risk = float(min(10, loan_to_income_ratio * 2))  # 5x income = 10 risk  
    lti_score = float(10 - lti_risk)
    
    # Narrative risk is already 0-10 scale (invert so lower is better) - ensure float
    narrative_score = float(10 - narrative_risk)
    
    # Calculate weighted composite score - ensure float
    composite_score = float(
        credit_score_normalized * credit_weight +
        dti_score * dti_weight +
        employment_score * employment_weight +
        lti_score * lti_weight +
        narrative_score * narrative_weight
    )
    
    # Determine risk category
    if composite_score >= 8.0:
        risk_category = "low_risk"
        recommendation = "approve"
    elif composite_score >= 6.5:
        risk_category = "moderate_risk" 
        recommendation = "approve_with_conditions"
    elif composite_score >= 5.0:
        risk_category = "high_risk"
        recommendation = "manual_review"
    else:
        risk_category = "very_high_risk"
        recommendation = "decline"
    
    return {
        "composite_score": float(round(composite_score, 2)),
        "risk_category": str(risk_category),
        "recommendation": str(recommendation),
        "credit_component": float(round(credit_score_normalized, 2)),
        "dti_component": float(round(dti_score, 2)),
        "employment_component": float(round(employment_score, 2)),
        "lti_component": float(round(lti_score, 2)),
        "narrative_component": float(round(narrative_score, 2))
    }

print("🔧 Custom UDF 'calculate_comprehensive_risk' registered")
print("   • Combines 5 risk factors with custom business logic")
print("   • Returns structured decision with component scores")
print("   • Integrates traditional underwriting with AI narrative analysis")

## 🚀 Step 4: Semantic-Enhanced Processing Pipeline

Execute the full pipeline combining AI analysis with custom business logic:

In [ ]:
# 🚀 Execute semantic + custom UDF pipeline
print("🚀 Starting semantic-enhanced risk assessment pipeline...")

# Step 1: AI narrative analysis
narrative_analysis = loan_applications.select(
    "*",
    fc.semantic.extract(
        "application_notes",
        RiskAssessment,
        model_alias="business_analyst"
    ).alias("narrative_assessment")
).cache()

print("✅ Step 1: Narrative risk analysis completed")

# Step 2: Calculate loan-to-income ratio and apply custom UDF
comprehensive_assessment = narrative_analysis.select(
    "*",
    (fc.col("loan_amount") / fc.col("annual_income")).alias("loan_to_income_ratio"),
    calculate_comprehensive_risk(
        fc.col("credit_score"),
        fc.col("debt_to_income"),
        fc.col("employment_years"),
        fc.col("loan_amount") / fc.col("annual_income"),
        narrative_analysis.narrative_assessment.narrative_risk_score
    ).alias("risk_analysis")
).cache()

print("✅ Step 2: Custom UDF risk calculation completed")

# Step 3: Extract results for analysis
final_results = comprehensive_assessment.select(
    "application_id",
    "applicant_name", 
    "credit_score",
    "annual_income",
    "loan_amount",
    "debt_to_income",
    comprehensive_assessment.narrative_assessment.employment_stability.alias("employment_stability"),
    comprehensive_assessment.narrative_assessment.narrative_risk_score.alias("narrative_risk"),
    comprehensive_assessment.risk_analysis.composite_score.alias("composite_score"),
    comprehensive_assessment.risk_analysis.risk_category.alias("risk_category"),
    comprehensive_assessment.risk_analysis.recommendation.alias("recommendation")
)

print("\n🏦 COMPREHENSIVE RISK ASSESSMENT RESULTS:")
final_results.show()

print("✅ Semantic + Custom UDF pipeline completed successfully!")

## 📊 Step 5: Advanced Business Intelligence

Analyze the combined semantic and custom logic results:

In [ ]:
# 📊 Advanced business intelligence analysis
print("📊 ADVANCED BUSINESS INTELLIGENCE DASHBOARD")
print("="*55)

# Recommendation distribution
recommendation_breakdown = final_results.group_by("recommendation").agg(
    fc.count("*").alias("count"),
    fc.avg("composite_score").alias("avg_score"),
    fc.avg("credit_score").alias("avg_credit_score")
).order_by(fc.desc("count"))

print("\n💼 LOAN RECOMMENDATION ANALYSIS:")
recommendation_breakdown.show()

# Risk category distribution
risk_breakdown = final_results.group_by("risk_category").agg(
    fc.count("*").alias("count"),
    fc.avg("narrative_risk").alias("avg_narrative_risk")
).order_by(fc.desc("count"))

print("\n🎯 RISK CATEGORY DISTRIBUTION:")
risk_breakdown.show()

# Detailed risk factor analysis
detailed_analysis = comprehensive_assessment.select(
    "application_id",
    "applicant_name",
    comprehensive_assessment.narrative_assessment.stability_factors.alias("stability_factors"),
    comprehensive_assessment.narrative_assessment.risk_factors.alias("risk_factors"),
    comprehensive_assessment.narrative_assessment.special_considerations.alias("special_considerations"),
    comprehensive_assessment.risk_analysis.recommendation.alias("recommendation")
)

print("\n🔍 DETAILED RISK FACTOR ANALYSIS:")
detailed_analysis.show()

# Calculate pipeline metrics
total_applications = loan_applications.count()
approved = final_results.filter(fc.col("recommendation") == "approve").count()
conditional_approved = final_results.filter(fc.col("recommendation") == "approve_with_conditions").count() 
manual_review = final_results.filter(fc.col("recommendation") == "manual_review").count()
declined = final_results.filter(fc.col("recommendation") == "decline").count()

# Use .agg() method and get first row values using correct QueryResult API
avg_scores = final_results.agg(
    fc.avg("composite_score").alias("avg_composite"),
    fc.avg("narrative_risk").alias("avg_narrative")
)
avg_results = avg_scores.collect()
# Use .data attribute to access the underlying data structure
polars_df = avg_results.data
avg_composite_score = polars_df.get_column("avg_composite")[0]
avg_narrative_risk = polars_df.get_column("avg_narrative")[0]

print(f"\n🎯 PORTFOLIO RISK METRICS:")
print(f"   • Total applications processed: {total_applications}")
print(f"   • Approved: {approved} ({approved/total_applications*100:.1f}%)")
print(f"   • Conditional approval: {conditional_approved} ({conditional_approved/total_applications*100:.1f}%)")
print(f"   • Manual review required: {manual_review} ({manual_review/total_applications*100:.1f}%)")
print(f"   • Declined: {declined} ({declined/total_applications*100:.1f}%)")
print(f"   • Average composite risk score: {avg_composite_score:.2f}/10")
print(f"   • Average narrative risk: {avg_narrative_risk:.2f}/10")

print(f"\n🛠️ CUSTOM UDF + SEMANTIC AI BENEFITS:")
print("   • Domain expertise: Business rules encoded in custom functions")
print("   • Semantic understanding: AI analyzes unstructured narrative data")
print("   • Comprehensive scoring: 5-factor risk model with AI enhancement")
print("   • Explainable decisions: Component scores provide transparency")
print("   • Scalable processing: Handle thousands of applications efficiently")
print("   • Consistent evaluation: Eliminate human bias in initial screening")
print("   • Risk segmentation: Automatic categorization for workflow routing")

print(f"\n🏆 PRODUCTION ADVANTAGES:")
print("   ✅ Combines traditional underwriting with AI insights")
print("   ✅ Extensible framework for domain-specific logic")
print("   ✅ Explainable AI with component-level scoring")
print("   ✅ Handles complex financial situations automatically")
print("   ✅ Reduces manual review workload")
print("   ✅ Consistent risk assessment at scale")
print("   ✅ Regulatory compliance through transparent scoring")

print(f"\n💡 BUSINESS IMPACT:")
print(f"   • Faster loan processing: Minutes instead of hours")
print(f"   • Reduced manual review: {(total_applications - manual_review)/total_applications*100:.1f}% automated")
print(f"   • Better risk assessment: Combines structured + unstructured data")
print(f"   • Improved customer experience: Faster decisions with clear explanations")
print(f"   • Enhanced compliance: Auditable decision process with component scores")

In [ ]:
session.stop()